# Markitdown Experiment

Experimenting with [MarkItDown](https://github.com/microsoft/markitdown) — Microsoft's utility that converts documents and images to Markdown for LLM pipelines.

Conversions covered, using files in `data/`:

| Section | Files | Converter |
|---|---|---|
| Doc | `doc/` | plain + OCR plugin cell + DOCX→PDF page-render pipeline |
| Image | `images/` | plain (metadata) + LLM (description / transcription) |
| PDFs | `pdfs/` | plain text-layer extraction + `md_llm` probe + scanned-PDF vision pipeline + OCR plugin |
| Scanned PDF | rendered PDF pages | PyMuPDF render → multimodal LLM (`gpt-6-luna`, reasoning effort `none`) |
| PPT | `powerpoints/` | plain + LLM alt-text + OCR plugin + PPTX→PDF page pipeline |
| Excel | `spreadsheets/` | plain (tables) + OCR plugin |

> Two LLM routes are used here:
> **1)** MarkItDown's built-in LLM support for **image files and pptx images** (`llm_client`/`llm_model` on the converter);
> **2)** the [`markitdown-ocr`](https://github.com/microsoft/markitdown#plugins) plugin, which OCRs images **embedded in pdf / docx / pptx / xlsx** via `MarkItDown(enable_plugins=True, ...)`.
> A scanned PDF has no text layer, so its pages are rendered with PyMuPDF and transcribed by the multimodal model directly — the plugin offers 300 DPI full-page fallback for the same scenario.

In [6]:
# Import libraries
import base64
import os
import shutil
import subprocess
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from markitdown import MarkItDown
from openai import OpenAI
import pymupdf  # renders PDF pages to PNG (used for the scanned-PDF section)

DATA = Path("data")
OUT = Path("out")
OUT.mkdir(exist_ok=True)

print("data dir ok:", DATA.is_dir())

data dir ok: True


In [7]:
# Setup llm and keys
load_dotenv()  # reads OPENAI_API_KEY from .env

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing - add it to the .env file next to this notebook.")

client = OpenAI(api_key=api_key, max_retries=3)

# Multimodal model used for image / scanned-page transcription.
# Override in .env with OPENAI_MODEL=<name> if the default is unavailable.
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
REASONING_EFFORT = "none"

# Shared transcription prompt for every LLM-powered conversion in this notebook.
OCR_PROMPT = (
    "Transcribe ALL text visible in this image into clean Markdown. "
    "Preserve headings with # markers, lists with -, and reproduce tables as "
    "GitHub-style Markdown tables. Keep the original language. Return only the Markdown."
)


class _CompletionsWithEffort:
    """Injects `reasoning_effort` into every chat.completions.create() call."""

    def __init__(self, completions, effort):
        self._completions = completions
        self._effort = effort

    def create(self, **kwargs):
        kwargs.setdefault("extra_body", {})
        kwargs["extra_body"]["reasoning_effort"] = self._effort
        return self._completions.create(**kwargs)

    def __getattr__(self, name):
        return getattr(self._completions, name)


class _ChatWithEffort:
    def __init__(self, chat, effort):
        self._chat = chat
        self.completions = _CompletionsWithEffort(chat.completions, effort)

    def __getattr__(self, name):
        return getattr(self._chat, name)


class ReasoningNoneClient:
    """Drop-in wrapper around an OpenAI client.

    MarkItDown and the markitdown-ocr plugin both call
    `llm_client.chat.completions.create(...)`, so hooking there forces
    reasoning_effort='none' for every conversion they perform.
    """

    def __init__(self, inner, effort="none"):
        self.chat = _ChatWithEffort(inner.chat, effort)
        self._inner = inner

    def __getattr__(self, name):
        return getattr(self._inner, name)


llm_client = ReasoningNoneClient(client, REASONING_EFFORT)

# 1) Plain converter: text/metadata extraction only, never calls the API.
md_plain = MarkItDown()
# 2) LLM converter: adds image descriptions / transcriptions via the multimodal model.
md_llm = MarkItDown(llm_client=llm_client, llm_model=MODEL)
# 3) Plugin converter: markitdown-ocr registers OCR-enhanced pdf/docx/pptx/xlsx
#    converters ahead of the built-ins when enable_plugins=True.
md_ocr = MarkItDown(
    enable_plugins=True,
    llm_client=llm_client,
    llm_model=MODEL,
    llm_prompt=OCR_PROMPT,
)

print("model:", MODEL, "| reasoning effort:", REASONING_EFFORT)
print("converters: md_plain (text only) | md_llm (vision images) | md_ocr (plugin OCR)")

model: gpt-6-luna | reasoning effort: none
converters: md_plain (text only) | md_llm (vision images) | md_ocr (plugin OCR)


In [8]:
# Shared helpers: run a conversion, preview it, save the full markdown
SUMMARY = []  # (source, converter kind, output chars)


def save_markdown(name, text):
    path = OUT / name
    path.write_text(text, encoding="utf-8")
    return path


def preview(text, chars=900):
    if len(text) <= chars:
        return text
    return text[:chars] + f"\n\n> ... {len(text) - chars:,} more characters (full text saved in `out/`)"


def convert_and_show(path, converter=md_plain, prompt=None, name=None, chars=900):
    """Convert a file, render a preview, and save the full markdown to out/."""
    path = Path(path)
    kwargs = {"llm_prompt": prompt} if prompt else {}
    result = converter.convert(str(path), **kwargs)
    text = result.markdown or ""
    kind = "ocr-plugin" if converter is md_ocr else ("llm" if converter is md_llm else "plain")
    out_path = save_markdown(name or (path.stem + ".md"), text)
    SUMMARY.append((str(path), kind, len(text)))
    display(Markdown(f"**`{path}`** -> `{out_path}` ({len(text):,} chars)\n\n---"))
    display(Markdown(preview(text, chars)))
    return text

## Doc

Word documents go through core preprocessing (styles, math, Mammoth HTML→Markdown), keeping headings, lists, and tables intact.

The `markitdown-ocr` plugin's enhanced DOCX converter OCRs embedded images and inserts each fragment inline where the image sat, wrapped as `*[Image OCR] … [End OCR]*`. The current article contains **no embedded images**, so its plugin run equals the plain output at zero API cost — flip the flag below once an image-bearing version is in place.

**Layout note:** DOCX stores no pagination — a layout engine (Word / LibreOffice) computes pages at render time, so `md.convert` never sees "pages". For exact page-level transcription like the scanned-PDF pipeline, export the DOCX to PDF first — see the **DOCX pages** subsection at the end of this section.

In [18]:
# Plain DOCX conversion
convert_and_show(DATA / "doc" / "multi_agent_workflows_ai_article.docx", chars=1100)

**`data\doc\multi_agent_workflows_ai_article.docx`** -> `out\multi_agent_workflows_ai_article.md` (9,108 chars)

---

**BEYOND A
SINGLE AI**

How Multi-Agent Workflows Turn Language Models into Collaborative Systems

*A practical introduction to agent roles, orchestration, shared context, and reliable execution*

**AI ENGINEERING FIELD NOTES**

# Introduction: From Chatbots to Coordinated Systems

A language model can summarize a report, write code, or answer a question. But many real-world tasks are not single-step problems. They require research, planning, execution, verification, and sometimes the ability to recover when something goes wrong. A multi-agent workflow organizes these activities into a coordinated process rather than asking one model to do everything at once.

The term “multi-agent” can sound more complicated than it needs to be. In practice, it often means assigning different responsibilities to separate model-driven components and defining how work moves between them. One component gathers evidence, another proposes a plan, a third performs a task, and a reviewer checks the result. The value comes not from the number of agents, but from the quality of the workflow.

# 1. What Is a 

> ... 8,008 more characters (full text saved in `out/`)

"**BEYOND A\nSINGLE AI**\n\nHow Multi-Agent Workflows Turn Language Models into Collaborative Systems\n\n*A practical introduction to agent roles, orchestration, shared context, and reliable execution*\n\n**AI ENGINEERING FIELD NOTES**\n\n# Introduction: From Chatbots to Coordinated Systems\n\nA language model can summarize a report, write code, or answer a question. But many real-world tasks are not single-step problems. They require research, planning, execution, verification, and sometimes the ability to recover when something goes wrong. A multi-agent workflow organizes these activities into a coordinated process rather than asking one model to do everything at once.\n\nThe term “multi-agent” can sound more complicated than it needs to be. In practice, it often means assigning different responsibilities to separate model-driven components and defining how work moves between them. One component gathers evidence, another proposes a plan, a third performs a task, and a reviewer checks

In [4]:
# LLM DOCX conversion
convert_and_show(
    DATA / "doc" / "perkembangan_transportasi_agustus_2026.docx",
    converter=md_llm,
    name="llm_perkembangan_transportasi_agustus_2026.md",
    chars=1400
)

**`data\doc\perkembangan_transportasi_agustus_2026.docx`** -> `out\llm_perkembangan_transportasi_agustus_2026.md` (13,011 chars)

---

![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)

***BADAN PUSAT STATISTIK KOTA PALU***

![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)

# ![](data:image/png;base64...) B. Perkembangan Transportasi Kota Palu

* Jumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu selama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan Juli 2026.
* Volume barang yang dibongkar melalui bandar udara tercatat 979.186 ton atau turun sebesar 8,65 persen, sementara itu barang yang dimuat tercatat 495.693 ton mengalami penurunan sebesar 37,43 persen dibandingkan Juli 2026.
* Pada Agustus 2026, Jumlah penumpang melalui Pelabuhan Pantoloan tercatat

8.016 orang.

* Volume barang yang dibongkar melalui Pelabuhan Pantoloan sebesar 10.653 ton, sementara volume barang yang dimuat tidak ada.
* Jumlah petikemas yang dibongkar me

> ... 11,611 more characters (full text saved in `out/`)

'![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)\n\n***BADAN PUSAT STATISTIK KOTA PALU***\n\n![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)![](data:image/png;base64...)\n\n# ![](data:image/png;base64...) B. Perkembangan Transportasi Kota Palu\n\n* Jumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu selama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan Juli 2026.\n* Volume barang yang dibongkar melalui bandar udara tercatat 979.1

### markitdown-ocr plugin (DOCX)

With `enable_plugins=True` in the setup cell, the plugin's docx converter takes over `md.convert()` — it OCRs embedded images inline in reading order and wraps results as `*[Image OCR] … [End OCR]*`. Pages without a text layer are auto-rendered at 300 DPI and sent whole to the vision model.

In [4]:
# Plugin OCR: one vision call per unique embedded image.
# NOTE: perkembangan_transportasi_agustus_2026.docx holds 85 unique images
# -> ~85 calls per run (vs ~7 with the DOCX page-render route below).
RUN_DOC_PLUGIN_OCR = True
if RUN_DOC_PLUGIN_OCR:
    convert_and_show(
        DATA / "doc" / "perkembangan_transportasi_agustus_2026.docx",
        converter=md_ocr,
        name="ocr_perkembangan_transportasi_agustus_2026.md",
        chars=1400,
    )

e:\NLP Learning\NLP-Learning\markitdown-experiment\.venv\Lib\site-packages\markitdown\converter_utils\docx\_images.py:72: RuntimeWarning: DOCX image OCR failed: Error code: 400 - {'error': {'message': 'Invalid prompt: your prompt was flagged as potentially violating our usage policy. Please try again with a different prompt: https://platform.openai.com/docs/guides/reasoning#advice-on-prompting', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_prompt'}}. Keeping the native image.
  fragment = self._render(image_stream, stream_info, **self._options)


**`data\doc\perkembangan_transportasi_agustus_2026.docx`** -> `out\ocr_perkembangan_transportasi_agustus_2026.md` (21,103 chars)

---

*[Image OCR]
Image
[End OCR]*

*[Image OCR]
The image contains no visible text.
[End OCR]*

*[Image OCR]
![Image](data:image/png;base64,)
[End OCR]*

*[Image OCR]

[End OCR]*

*[Image OCR]
This image appears blank.
[End OCR]*

*[Image OCR]
# Sample Input

# Sample Output
[End OCR]*

*[Image OCR]
Image
[End OCR]*

*[Image OCR]
The image contains no visible text.
[End OCR]*

![](data:image/png;base64...)

*[Image OCR]
#】【。
[End OCR]*

*[Image OCR]
I can’t identify any text in the image.
[End OCR]*

***BADAN PUSAT STATISTIK KOTA PALU***

*[Image OCR]
No text is visible in the image.
[End OCR]*

*[Image OCR]
The image contains no visible text.
[End OCR]*

*[Image OCR]
<!-- The image contains no visible text. -->
[End OCR]*

*[Image OCR]
The image appears blank.
[End OCR]*

*[Image OCR]
# (text too small to read)
[End OCR]*

*[Image OCR]
# Text Visible in Image

- The image is blank.
[End OCR]*

*[Image OCR]
![Image](attachment)
[End OCR]*

*[Image OCR]
![](data:image/png;base64,iVBORw0KGg

> ... 19,703 more characters (full text saved in `out/`)

### DOCX pages → LLM (via PDF export)

Because pagination lives in the layout engine rather than the file, the page-image trick from the scanned-PDF section needs one extra hop: **export DOCX → PDF with Word or LibreOffice**, then the exact render → OCR → stitch flow applies.

`docx_to_pdf()` below auto-detects the engine: MS Word via COM (`docx2pdf`) first, LibreOffice headless (`soffice --convert-to pdf`) second, and a clear error with a manual-export fallback path if neither exists.

For the target file the PDF twin already produced excellent page-level transcripts (Scanned-PDF section) — this reproduces that straight from the DOCX (~1 call per page ≈ 7 calls). Needs `vision_to_markdown()` from the Scanned-PDF section — run that part first.

In [ ]:
# DOCX -> PDF export with auto-detected layout engine (Word first, LibreOffice fallback)
def docx_to_pdf(docx_path, out_dir=None):
    docx_path = Path(docx_path)
    out_dir = Path(out_dir or OUT / "docx_pdf")
    out_dir.mkdir(exist_ok=True, parents=True)
    dest = out_dir / (docx_path.stem + ".pdf")

    word_error = None
    try:
        from docx2pdf import convert  # MS Word COM automation (Windows + Office)

        convert(str(docx_path), str(dest))
        if dest.exists():
            return dest
    except Exception as e:  # no Word / COM error -> try LibreOffice next
        word_error = e

    soffice = shutil.which("soffice") or shutil.which("soffice.exe")
    if soffice:
        subprocess.run(
            [soffice, "--headless", "--norestore", "--convert-to", "pdf", "--outdir", str(out_dir), str(docx_path)],
            check=True,
            capture_output=True,
            timeout=600,
        )
        if dest.exists():
            return dest

    word_note = (
        f" (docx2pdf/COM error: {type(word_error).__name__}: {word_error})"
        if word_error
        else " (docx2pdf/COM ran but produced no PDF)"
    )
    raise RuntimeError(
        f"No layout engine available to render '{docx_path.name}' to PDF.{word_note}\n"
        "- Option A: install MS Word\n"
        "- Option B: install LibreOffice (put soffice.exe on PATH)\n"
        "- Option C: export the PDF yourself (Word / LibreOffice / Google Docs)\n"
        f"  and save it as: {dest} - the next cell picks it up."
    )


DOCX_TARGET = DATA / "doc" / "perkembangan_transportasi_agustus_2026.docx"
docx_pdf_path = docx_to_pdf(DOCX_TARGET)
print("engine used: auto-detected -> exported", docx_pdf_path.name, f"({docx_pdf_path.stat().st_size / 1024:.0f} KB)")

In [ ]:
# Render the exported PDF's pages, OCR each with the direct vision call
# (reasoning_effort="none"), and stitch -> out/scanned_docx_<stem>.md
DOCX_PAGES_DIR = OUT / "docx_pages"
DOCX_PAGES_DIR.mkdir(exist_ok=True, parents=True)

doc = pymupdf.open(docx_pdf_path)
docx_pages = []
for i, page in enumerate(doc, start=1):
    png = DOCX_PAGES_DIR / f"{docx_pdf_path.stem}_p{i:02d}.png"
    page.get_pixmap(dpi=150).save(png)
    docx_pages.append(png)
doc.close()

docx_pages_md = []
for png in docx_pages:
    print("ocr:", png.name, "...", end=" ")
    t0 = time.perf_counter()
    text = vision_to_markdown(png).strip()
    print(f"{len(text):,} chars in {time.perf_counter() - t0:.1f}s")
    docx_pages_md.append(f"<!-- page {png.stem[-2:]} -->\n\n{text}")

docx_stitched = "\n\n---\n\n".join(docx_pages_md)
docx_stitched_path = save_markdown(f"scanned_docx_{docx_pdf_path.stem}.md", docx_stitched)
SUMMARY.append((str(DOCX_TARGET), "llm-scanned (docx via pdf)", len(docx_stitched)))
display(Markdown(f"**`{DOCX_TARGET.name}`** (via PDF export) -> `{docx_stitched_path.name}` ({len(docx_stitched):,} chars)\n\n---"))
display(Markdown(preview(docx_stitched, 900)))

## Image

Image handling comes in two flavors:

1. **Plain** (`md_plain`) — metadata only (EXIF, dimensions, ...). Without `exiftool` installed this is nearly empty output.
2. **LLM** (`md_llm`) — sends the image to the configured multimodal model. The default prompt asks for a *detailed caption* (`"Write a detailed caption for this image."`); pass `prompt=...` to make it transcribe text instead.

In [4]:
# Plain conversion: no API calls, so image files yield almost no text
convert_and_show(DATA / "images" / "sample_text.jpg")
convert_and_show(DATA / "images" / "sample_image_1.png")

**`data\images\sample_text.jpg`** -> `out\sample_text.md` (0 chars)

---

**`data\images\sample_image_1.png`** -> `out\sample_image_1.md` (0 chars)

---

''

In [5]:
# LLM conversion: one API call per image, transcription prompt, reasoning_effort="none"
for img in [
    "sample_text.jpg",     # simple typed text -> straight transcription
    "sample_image_1.png",  # document-like page (header, table, figure, checklist)
    "page_2.png",          # scanned page from a Japanese engineering document
]:
    convert_and_show(
        DATA / "images" / img,
        converter=md_llm,
        prompt=OCR_PROMPT,
        name=f"llm_{Path(img).stem}.md",
    )

**`data\images\sample_text.jpg`** -> `out\llm_sample_text.md` (65 chars)

---


# Description:
The quick brown fox jumped over the 5 lazy dogs!


**`data\images\sample_image_1.png`** -> `out\llm_sample_image_1.md` (1,409 chars)

---


# Description:
This is the header of the document.

# This is title

## 1. Text

Latin refers to an ancient Italic language originating in the region of Latium in ancient Rome.

## 2. Page Objects

### 2.1 Table

Here's a sample table below, designed to be simple for easy understand and quick reference.

| Name | Corp | Remark |
|---|---|---|
| Foo |  |  |
| Bar | Microsoft | Dummy |

*Table 1: This is a dummy table*

### 2.2. Figure

*Figure 1: Here is a figure with text*

Values: 100, 200, 300, 400, 500
Jan: 200
Feb: 300
Mar: 400
Apr: 450
May: 350
Jun: 250
Month

## 3. Others

AI Document Intelligence is an AI service that applies advanced machine learning to extract text, key-value pairs, tables, and structures from documents automatically and accurately:

- ☑ clear
- ☑ precise
- ☐ vague
- ☑ coherent
- ☐ Incomprehensible

Turn documents into usable data and shift your focus to acting

> ... 509 more characters (full text saved in `out/`)

**`data\images\page_2.png`** -> `out\llm_page_2.md` (1,556 chars)

---


# Description:
[目次へ戻る](#)

| 車両情報　BMC情報 | <u>FRバンパーFR (UPR) + FR CB (UPR)</u> |
|---|---|
| Brand | 部品点数 |
|  | Quantity of Parts |
|  | Total Weight (kg) |
| Cadillac Lyriq Debut Edition 2023 | 1 | 6.84 |
| Smart #1 Premium 2023 | 5 | 13.34 |
| Volkswagen Golf VIII 1.5 eTSI Style 1st Edition II 2020 | 1 | 7.98 |
| Volkswagen Polo 1.0 TSi Highline 2018 | 1 | 7.99 |
| Volkswagen Tiguan 2.0 TDi 4Motion Highline 2016 | 1 | 9.56 |
| Volkswagen Passat Variant 2.0 TDi SCR Highline 2015 | 1 | 7.65 |
| Volkswagen Golf VII 1.4 TSi Comfortline 2013 | 1 | 8.68 |
| Volkswagen Up! 1.0 Take Up! 2012 | 1 | 6.13 |
| Volkswagen Touran 2.0 TDi Highline 2011 | 1 | 7.41 |
| Audi Q4 e-tron 50 quattro 2021 | 1 | 19.31 |
| Hyundai Tucson Plug-in Prime 2021 | 1 | 7.81 |
| Hyundai Ioniq 1.6 Plug-in 2017 | 3 | 6.40 |
| Hyundai i30 1.0 T-GDi Edition One 2016 | 1 | 7.07 |
| Hyundai i10 1.2 Pack Evidence 2014 | 1 |

> ... 656 more characters (full text saved in `out/`)

## PDFs

The built-in PDF converter extracts the **text layer** with `pdfminer` — no API calls, works well for digital PDFs. Converting the whole folder and previewing each result:

In [6]:
# Convert every PDF with the plain text-layer converter, save + preview each
pdf_results = {}
for pdf in sorted((DATA / "pdfs").glob("*.pdf")):
    text = convert_and_show(pdf, name=pdf.stem + ".md", chars=350)
    pdf_results[pdf.name] = text

**`data\pdfs\docling_technical_report.pdf`** -> `out\docling_technical_report.md` (84,201 chars)

---

Docling Technical Report
Version1.0
ChristophAuer MaksymLysak AhmedNassar MicheleDolfi NikolaosLivathinos
PanosVagenas CesarBerrospiRamis MatteoOmenetti FabianLindlbauer
KasperDinkla LokeshMishra YusikKim ShubhamGupta RafaelTeixeiradeLima
ValeryWeber LucasMorin IngmarMeijer ViktorKuropiatnyk PeterW.J.Staar
AI4KGroup,IBMResearch
Ru¨schlikon,Switzerl

> ... 83,851 more characters (full text saved in `out/`)

**`data\pdfs\FinancialStatement-2025-III-BBRI.pdf`** -> `out\FinancialStatement-2025-III-BBRI.md` (123,596 chars)

---

Nomor Surat B.5496-DIR/FMA/10/2025
Nama Emiten PT Bank Rakyat Indonesia (Persero) Tbk
Kode Emiten BBRI
Perihal Penyampaian Laporan Keuangan Interim Yang Tidak Diaudit
Perseroan dengan ini menyampaikan laporan keuangan untuk periode 9 Bulan yang berakhir pada 30/09/2025 dengan ikhtisar sebagai berikut :
Informasi mengenai anak perusahaan Perseroan s

> ... 123,246 more characters (full text saved in `out/`)

**`data\pdfs\long_table_pdf.pdf`** -> `out\long_table_pdf.md` (51,532 chars)

---

...

Design and build
accessible PDF tables
Sample tables
Table 1
| Column header (TH)  | Column header (TH)  | Column header (TH)  |     |
| ------------------- | ------------------- | ------------------- | --- |
| Row header (TH)     | Data cell (TD)      | Data cell (TD)      |     |
| Row header(TH)      | Data cell (TD)      | Data cell (TD)      |

> ... 22,094 more characters (full text saved in `out/`)

**`data\pdfs\Product Strategy & Decision Handbook — Atlas Pay.pdf`** -> `out\Product Strategy & Decision Handbook — Atlas Pay.md` (8,633 chars)

---

Product Strategy & Decision Handbook

Product: AtlasPay

Company: Atlas Fintech Solutions

Last Updated: 12 January 2026

Document Owner: Product Strategy Team

Audience: Product Managers, Product Analysts, Data Analysts, Engineering Leads, Business Stakeholders

1. Purpose of This Document

This handbook serves as the single source of truth for At

> ... 8,283 more characters (full text saved in `out/`)

**`data\pdfs\sustainable_urban_mobility_trends_2025.pdf`** -> `out\sustainable_urban_mobility_trends_2025.md` (3,389 chars)

---

Sustainable Urban
Mobility Trends 2025
This presentation offers urban planners insights into sustainable urban
mobility. We will explore key trends and strategies. Our focus areas include
EV adoption, micromobility, and smart infrastructure. Our goal is to guide
future urban development.
by Muhammad Cikal Merdeka

Key Challenges in Urban Mobility
|

> ... 3,039 more characters (full text saved in `out/`)

In [7]:
# Closer look at a text PDF whose value is almost entirely data tables:
# markitdown has to reconstruct the pipe-table structure from the text layer.
convert_and_show(
    DATA / "pdfs" / "long_table_pdf.pdf",
    name="focus_long_table.md",
    chars=1400,
)

**`data\pdfs\long_table_pdf.pdf`** -> `out\focus_long_table.md` (51,532 chars)

---

REKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS
(FORMULIR LAPORAN)

SIMPANG KUALA TANJUNG KE INALUM
POS 1 - AKSES ROAD INALUM (PRM TJ GADING)

15-07-2019

Tanggal :
Nama : MOH SUHERI

ANDRIANUS

5a

5b

6a

6b

7a

7b

7c

REKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS
(FORMULIR LAPORAN)

INALUM ke SIMPANG KUALA TANJUNG
POS 1 - AKSES ROAD INALUM (PRM TJ GADING)

15-07-2019

Tanggal :
Nama : JOKO SAMUDRO

Arah :

Hari 1

Pukul

06.00 - 06.15

06.15 - 06.30

06.30 - 06.45

06.45 - 07.00

07.00 - 07.15

07.15 - 07.30

07.30 - 07.45

07.45 - 08.00

08.00 - 08.15

08.15 - 08.30

08.30 - 08.45

08.45 - 09.00

09.00 - 09.15

09.15 - 09.30

09.30 - 09.45

09.45 - 10.00

10.00 - 10.15

10.15 - 10.30

10.30 - 10.45

10.45 - 11.00

11.00 - 11.15

11.15 - 11.30

11.30 - 11.45

11.45 - 12.00

12.00 - 12.15

12.15 - 12.30

12.30 - 12.45
12.45 - 13.00
13.00 - 13.15

13.15 - 13.30

13.30 - 13.45
13.45 - 14.00

14.00 - 14.15
14.00 - 14.15
14.15 - 14.30
14.30 - 14.45
14.45 - 15.00
15.00 - 15.15
15.15 - 15.30
15.30 - 15.45
15.45 - 16.00

16.00 - 16.15

16.15 - 16.30
16.30 - 16.45
16.45 - 17.00
17.00 - 17.15
17.15 - 17.30
17.30 - 17.45
17.45 - 18.00
18.00 - 18.15
18.15 - 18.30
18.30 - 18.45

18.45 - 19.00
19.00 - 19.15
19.15 - 19.30
19.30 - 19.45
19.45 - 20.00
20.00 - 20.15
20.15 - 20.30
20.30 - 20.45
20.45 - 21.00
21.00 - 21.15
21.15 - 21.30
21.30 - 21.45
21.45 - 22.00
22.00 - 22.15
22.15 - 22.3

> ... 50,132 more characters (full text saved in `out/`)

'REKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS\n(FORMULIR LAPORAN)\n\nSIMPANG KUALA TANJUNG KE INALUM\nPOS 1 - AKSES ROAD INALUM (PRM TJ GADING)\n\n15-07-2019\n\nTanggal :\nNama : MOH SUHERI\n\nANDRIANUS\n\n5a\n\n5b\n\n6a\n\n6b\n\n7a\n\n7b\n\n7c\n\nREKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS\n(FORMULIR LAPORAN)\n\nINALUM ke SIMPANG KUALA TANJUNG\nPOS 1 - AKSES ROAD INALUM (PRM TJ GADING)\n\n15-07-2019\n\nTanggal :\nNama : JOKO SAMUDRO\n\nArah :\n\nHari 1\n\nPukul\n\n06.00 - 06.15\n\n06.15 - 06.30\n\n06.30 - 06.45\n\n06.45 - 07.00\n\n07.00 - 07.15\n\n07.15 - 07.30\n\n07.30 - 07.45\n\n07.45 - 08.00\n\n08.00 - 08.15\n\n08.15 - 08.30\n\n08.30 - 08.45\n\n08.45 - 09.00\n\n09.00 - 09.15\n\n09.15 - 09.30\n\n09.30 - 09.45\n\n09.45 - 10.00\n\n10.00 - 10.15\n\n10.15 - 10.30\n\n10.30 - 10.45\n\n10.45 - 11.00\n\n11.00 - 11.15\n\n11.15 - 11.30\n\n11.30 - 11.45\n\n11.45 - 12.00\n\n12.00 - 12.15\n\n12.15 - 12.30\n\n12.30 - 12.45\n12.45 - 13.00\n13.00 - 13.15\n\n13.15 - 13.30\n\n13.30 - 13.45\n13

### Built-in LLM route directly on PDFs?

Shortcut worth testing: `convert_and_show(..., converter=md_llm)` on PDFs that contain embedded images. Result: **it runs, but does nothing extra** — in markitdown 0.1.8 the built-in LLM hooks live only in the **image** and **PPTX** converters. The core PDF converter extracts the text layer (pdfplumber/pdfminer) and ignores the `llm_client` / `llm_model` / `llm_prompt` kwargs forwarded to it (e.g. `perkembangan_transportasi_agustus_2026.pdf` holds 12 unique embedded images, `docling_technical_report.pdf` 30 — still zero model calls).

The cell below proves that empirically: output should come out **identical to `md_plain`** with **no API cost**. For embedded-PDF-images the routes that actually reach the model are the 300-DPI pipeline (below) and the `markitdown-ocr` plugin.

In [4]:
# md_llm on PDFs: expect IDENTICAL output to md_plain and zero API calls
# (verified in source: PdfConverter.convert() never reads the llm_* kwargs).
for pdf_name in [
    "pdf_with_tables_example.pdf",                # 0 embedded images
    "perkembangan_transportasi_agustus_2026.pdf",  # 12 unique images -> ignored anyway
    "docling_technical_report.pdf",                # 30 unique images -> ignored anyway
]:
    pdf_path = DATA / "pdfs" / pdf_name
    llm_text = convert_and_show(
        pdf_path,
        converter=md_llm,
        name=f"llm_pdf_{Path(pdf_name).stem}.md",
        chars=350,
    )
    plain_text = (md_plain.convert(str(pdf_path)).markdown or "").strip()
    verdict = "identical" if llm_text.strip() == plain_text else "DIFFERENT"
    print(f"md_llm vs md_plain on {pdf_name} -> {verdict}")

**`data\pdfs\pdf_with_tables_example.pdf`** -> `out\llm_pdf_pdf_with_tables_example.md` (22,444 chars)

---

Design and build
accessible PDF tables
Sample tables
Table 1
| Column header (TH)  | Column header (TH)  | Column header (TH)  |     |
| ------------------- | ------------------- | ------------------- | --- |
| Row header (TH)     | Data cell (TD)      | Data cell (TD)      |     |
| Row header(TH)      | Data cell (TD)      | Data cell (TD)      |

> ... 22,094 more characters (full text saved in `out/`)

md_llm vs md_plain on pdf_with_tables_example.pdf -> identical


**`data\pdfs\perkembangan_transportasi_agustus_2026.pdf`** -> `out\llm_pdf_perkembangan_transportasi_agustus_2026.md` (11,392 chars)

---

BADAN PUSAT STATISTIK
KOTA PALU
B. Perkembangan Transportasi Kota Palu
 Jumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu
selama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan
Juli 2026.
 Volume barang yang dibongkar melalui bandar udara tercatat 979.186 ton atau
turun sebesar 8,65 persen, sem

> ... 11,042 more characters (full text saved in `out/`)

md_llm vs md_plain on perkembangan_transportasi_agustus_2026.pdf -> identical


**`data\pdfs\docling_technical_report.pdf`** -> `out\llm_pdf_docling_technical_report.md` (84,201 chars)

---

Docling Technical Report
Version1.0
ChristophAuer MaksymLysak AhmedNassar MicheleDolfi NikolaosLivathinos
PanosVagenas CesarBerrospiRamis MatteoOmenetti FabianLindlbauer
KasperDinkla LokeshMishra YusikKim ShubhamGupta RafaelTeixeiradeLima
ValeryWeber LucasMorin IngmarMeijer ViktorKuropiatnyk PeterW.J.Staar
AI4KGroup,IBMResearch
Ru¨schlikon,Switzerl

> ... 83,851 more characters (full text saved in `out/`)

md_llm vs md_plain on docling_technical_report.pdf -> identical


### Scanned PDF → LLM (vision OCR)

For a **scanned** PDF the text layer is missing, so the plain conversion returns (almost) nothing. Pipeline instead:

1. **Render** pages to PNG with `pymupdf` (raise `dpi=` for small print).
2. **Transcribe** each page image with the multimodal model (`gpt-6-luna`, `reasoning_effort="none"`) via direct vision calls using the shared `OCR_PROMPT`.
3. **Stitch** the per-page markdown into one document per PDF.

The cells below loop over three targets — ≈27 vision calls total: `pdf_with_tables_example.pdf` (11 pages) and `perkembangan_transportasi_agustus_2026.pdf` (7 pages) have text layers too, so their runs double as a quality comparison against the plain and plugin routes; `docling_technical_report.pdf` (9 pages) is dense text + diagrams.

> The `markitdown-ocr` plugin (subsection below) does the equivalent through `md.convert()` itself — its PDF converter even detects text-free pages and OCRs a 300 DPI render automatically.

In [8]:
# 1) Render every target PDF's pages to PNG
SCAN_PDFS = [
    "pdf_with_tables_example.pdf",                # 11 pages, tables (text layer too -> comparison case)
    "perkembangan_transportasi_agustus_2026.pdf",  # 7 pages, 12 embedded images
    "docling_technical_report.pdf",                # 9 pages, dense text + figures
]
PAGES_DIR = OUT / "scan_pages"
PAGES_DIR.mkdir(exist_ok=True)

rendered_by_pdf = {}
for pdf_name in SCAN_PDFS:
    pdf_path = DATA / "pdfs" / pdf_name
    doc = pymupdf.open(pdf_path)
    pages = []
    for i, page in enumerate(doc, start=1):
        png = PAGES_DIR / f"{pdf_path.stem}_p{i:02d}.png"
        page.get_pixmap(dpi=150).save(png)
        pages.append(png)
    doc.close()
    rendered_by_pdf[pdf_name] = pages
    print(f"{pdf_name}: rendered {len(pages)} pages -> {PAGES_DIR}")

pdf_with_tables_example.pdf: rendered 11 pages -> out\scan_pages
perkembangan_transportasi_agustus_2026.pdf: rendered 7 pages -> out\scan_pages
docling_technical_report.pdf: rendered 9 pages -> out\scan_pages


In [9]:
# 2) Transcribe each rendered page with the multimodal model (direct API call,
#    reasoning_effort="none"), then 3) stitch everything per PDF into one document.
def vision_to_markdown(image_path, prompt=OCR_PROMPT, *, model=MODEL, effort=REASONING_EFFORT):
    b64 = base64.b64encode(Path(image_path).read_bytes()).decode("utf-8")
    ext = Path(image_path).suffix.lstrip(".").lower()
    mime = "jpeg" if ext == "jpg" else ext
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}},
                ],
            }
        ],
        extra_body={"reasoning_effort": effort},
    )
    return response.choices[0].message.content or ""


for pdf_name, pages in rendered_by_pdf.items():
    pages_md = []
    for png in pages:
        print("ocr:", pdf_name, "/", png.name, "...", end=" ")
        t0 = time.perf_counter()
        text = vision_to_markdown(png).strip()
        print(f"{len(text):,} chars in {time.perf_counter() - t0:.1f}s")
        pages_md.append(f"<!-- page {png.stem[-2:]} -->\n\n{text}")

    stitched = "\n\n---\n\n".join(pages_md)
    stitched_path = save_markdown("scanned_" + Path(pdf_name).stem + ".md", stitched)
    SUMMARY.append((str(DATA / "pdfs" / pdf_name), "llm-scanned", len(stitched)))
    display(Markdown(f"**`{pdf_name}`** -> `{stitched_path.name}` ({len(stitched):,} chars)\n\n---"))
    display(Markdown(preview(stitched, 900)))

ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p01.png ... 1,223 chars in 5.0s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p02.png ... 1,150 chars in 7.5s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p03.png ... 719 chars in 7.4s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p04.png ... 1,466 chars in 4.8s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p05.png ... 1,269 chars in 4.3s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p06.png ... 1,394 chars in 4.0s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p07.png ... 984 chars in 4.4s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p08.png ... 1,127 chars in 4.4s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p09.png ... 1,249 chars in 6.6s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p10.png ... 737 chars in 3.4s
ocr: pdf_with_tables_example.pdf / pdf_with_tables_example_p11.png ... 709 chars in 4.4s


**`pdf_with_tables_example.pdf`** -> `scanned_pdf_with_tables_example.md` (12,295 chars)

---

<!-- page 01 -->

# Design and build
# accessible PDF tables

## Sample tables

### Table 1

| Column header (TH) | Column header (TH) | Column header (TH) |
|---|---|---|
| **Row header (TH)** | Data cell (TD) | Data cell (TD) |
| **Row header(TH)** | Data cell (TD) | Data cell (TD) |

### Table 2: example of footnotes referenced from within a table

| Expenditure by function £ million |  | 2009/10 | 2010/11 <sup>1</sup> |
|---|---|---:|---:|
| **Policy functions** | **Financial** | 22.5 | 30.57 |
|  | **Information <sup>2</sup>** | 10.2 | 14.8 |
|  | **Contingency** | 2.6 | 1.2 |
| **Remunerated functions** | **Agency services <sup>3</sup>** | 44.7 | 35.91 |
|  | **Payments** | 22.41 | 19.88 |
|  | **Banking** | 22.90 | 44.23 |
|  | **Other** | 12.69 | 10.32 |

(1) Provisional total as of publication date.  
(2) Costs associated with on-going information programmes.  
(3) From the mana

> ... 11,395 more characters (full text saved in `out/`)

ocr: perkembangan_transportasi_agustus_2026.pdf / perkembangan_transportasi_agustus_2026_p01.png ... 940 chars in 3.9s
ocr: perkembangan_transportasi_agustus_2026.pdf / perkembangan_transportasi_agustus_2026_p02.png ... 2,755 chars in 5.6s
ocr: perkembangan_transportasi_agustus_2026.pdf / perkembangan_transportasi_agustus_2026_p03.png ... 2,628 chars in 8.1s
ocr: perkembangan_transportasi_agustus_2026.pdf / perkembangan_transportasi_agustus_2026_p04.png ... 2,383 chars in 8.5s
ocr: perkembangan_transportasi_agustus_2026.pdf / perkembangan_transportasi_agustus_2026_p05.png ... 830 chars in 3.7s
ocr: perkembangan_transportasi_agustus_2026.pdf / perkembangan_transportasi_agustus_2026_p06.png ... 1,420 chars in 5.4s
ocr: perkembangan_transportasi_agustus_2026.pdf / perkembangan_transportasi_agustus_2026_p07.png ... 613 chars in 2.7s


**`perkembangan_transportasi_agustus_2026.pdf`** -> `scanned_perkembangan_transportasi_agustus_2026.md` (11,737 chars)

---

<!-- page 01 -->

BADAN PUSAT STATISTIK  
KOTA PALU

# BERITA  
# RESMI  
# STATISTIK

## B. Perkembangan Transportasi Kota Palu

- Jumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu selama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan Juli 2026.
- Volume barang yang dibongkar melalui bandar udara tercatat 979.186 ton atau turun sebesar 8,65 persen, sementara itu barang yang dimuat tercatat 495.693 ton mengalami penurunan sebesar 37,43 persen dibandingkan Juli 2026.
- Pada Agustus 2026, Jumlah penumpang melalui Pelabuhan Pantoloan tercatat 8.016 orang.
- Volume barang yang dibongkar melalui Pelabuhan Pantoloan sebesar 10.653 ton, sementara volume barang yang dimuat tidak ada.
- Jumlah petikemas yang dibongkar melalui pelabuhan pantoloan sebesar 4.527 box sementara jumlah petikemas yang dimuat sebesar 1.570
- Frekuensi pelayaran yang b

> ... 10,837 more characters (full text saved in `out/`)

ocr: docling_technical_report.pdf / docling_technical_report_p01.png ... 2,214 chars in 6.5s
ocr: docling_technical_report.pdf / docling_technical_report_p02.png ... 3,541 chars in 6.4s
ocr: docling_technical_report.pdf / docling_technical_report_p03.png ... 3,210 chars in 6.0s
ocr: docling_technical_report.pdf / docling_technical_report_p04.png ... 4,149 chars in 7.5s
ocr: docling_technical_report.pdf / docling_technical_report_p05.png ... 3,811 chars in 9.3s
ocr: docling_technical_report.pdf / docling_technical_report_p06.png ... 3,165 chars in 8.5s
ocr: docling_technical_report.pdf / docling_technical_report_p07.png ... 7,252 chars in 17.3s
ocr: docling_technical_report.pdf / docling_technical_report_p08.png ... 9,218 chars in 25.7s
ocr: docling_technical_report.pdf / docling_technical_report_p09.png ... 4,352 chars in 16.3s


**`docling_technical_report.pdf`** -> `scanned_docling_technical_report.md` (41,130 chars)

---

<!-- page 01 -->

# Docling Technical Report

Version 1.0

Christoph Auer Maksy m Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos  
Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer  
Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima  
Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar

AI4K Group, IBM Research  
Rüschlikon, Switzerland

## Abstract

This technical report introduces *Docling*, an easy to use, self-contained, MIT-licensed open-source package for PDF document conversion. It is powered by state-of-the-art specialized AI models for layout analysis (DocLayNet) and table structure recognition (TableFormer), and runs efficiently on commodity hardware in a small resource budget. The code interface allows for easy extensibility and addition of new features and models.

# 1 Introduction

Converting PDF docu

> ... 40,230 more characters (full text saved in `out/`)

### Alternative: `md_llm` on the rendered pages

Fair question — since the renders are image files, why not just `convert_and_show(..., converter=md_llm)`? It **works**: the image converter forwards our `OCR_PROMPT` and the effort wrapper, at the same one-call-per-page cost. The difference is shape, not capability: the image converter is a *description* path — every answer comes back wrapped under a `# Description:` heading (and EXIF metadata would be interleaved on machines with exiftool), so the stitch step would need post-processing, and the raw response stays hidden. `vision_to_markdown()` keeps per-page transcripts verbatim, makes the request shape explicit, and times each call. Same page below, both become visible:

In [10]:
rendered_by_pdf

{'pdf_with_tables_example.pdf': [WindowsPath('out/scan_pages/pdf_with_tables_example_p01.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p02.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p03.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p04.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p05.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p06.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p07.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p08.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p09.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p10.png'),
  WindowsPath('out/scan_pages/pdf_with_tables_example_p11.png')],
 'perkembangan_transportasi_agustus_2026.pdf': [WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p01.png'),
  WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p02.png'),
  WindowsPath('out/scan_pages/perkembangan_tran

In [11]:
rendered_by_pdf["perkembangan_transportasi_agustus_2026.pdf"]

[WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p01.png'),
 WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p02.png'),
 WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p03.png'),
 WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p04.png'),
 WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p05.png'),
 WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p06.png'),
 WindowsPath('out/scan_pages/perkembangan_transportasi_agustus_2026_p07.png')]

In [12]:
# One rendered page through the image-converter route instead of the direct call:
alt_page = rendered_by_pdf["perkembangan_transportasi_agustus_2026.pdf"][0]
convert_and_show(
    alt_page,
    converter=md_llm,
    prompt=OCR_PROMPT,
    name=f"alt_{alt_page.stem}.md",
    chars=700,
)

**`out\scan_pages\perkembangan_transportasi_agustus_2026_p01.png`** -> `out\alt_perkembangan_transportasi_agustus_2026_p01.md` (951 chars)

---


# Description:
BADAN PUSAT STATISTIK
KOTA PALU

# BERITA
# RESMI
# STATISTIK

## B. Perkembangan Transportasi Kota Palu

- Jumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu selama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan Juli 2026.
- Volume barang yang dibongkar melalui bandar udara tercatat 979.186 ton atau turun sebesar 8,65 persen, sementara itu barang yang dimuat tercatat 495.693 ton mengalami penurunan sebesar 37,43 persen dibandingkan Juli 2026.
- Pada Agustus 2026, Jumlah penumpang melalui Pelabuhan Pantoloan tercatat 8.016 orang.
- Volume barang yang dibongkar melalui Pelabuhan Pantoloan sebesar 10.653 ton, sementara volume

> ... 251 more characters (full text saved in `out/`)

'\n# Description:\nBADAN PUSAT STATISTIK\nKOTA PALU\n\n# BERITA\n# RESMI\n# STATISTIK\n\n## B. Perkembangan Transportasi Kota Palu\n\n- Jumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu selama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan Juli 2026.\n- Volume barang yang dibongkar melalui bandar udara tercatat 979.186 ton atau turun sebesar 8,65 persen, sementara itu barang yang dimuat tercatat 495.693 ton mengalami penurunan sebesar 37,43 persen dibandingkan Juli 2026.\n- Pada Agustus 2026, Jumlah penumpang melalui Pelabuhan Pantoloan tercatat 8.016 orang.\n- Volume barang yang dibongkar melalui Pelabuhan Pantoloan sebesar 10.653 ton, sementara volume barang yang dimuat tidak ada.\n- Jumlah petikemas yang dibongkar melalui pelabuhan pantoloan sebesar 4.527 box sementara jumlah petikemas yang dimuat sebesar 1.570\n- Frekuensi pelayaran yang beroperasi di seluruh wilayah Kota Palu, tercatat 486 kali.\n'

### markitdown-ocr plugin (PDF)

With `enable_plugins=True` in the setup cell, the plugin's PDF converter takes over `md.convert()` — it OCRs embedded images inline in reading order and wraps results as `*[Image OCR] … [End OCR]*`. Pages without a text layer are auto-rendered at 300 DPI and sent whole to the vision model.

The mobility PDF holds **14 unique embedded images** → that many vision calls per run (repeated bytes are recognized once). Prefer `docling technical report` for diagram-heavy output (~30 calls) by changing the path below.

In [5]:
# ~14 vision calls (14 unique embedded images; repeated bytes are recognized once).
convert_and_show(
    DATA / "pdfs" / "sustainable_urban_mobility_trends_2025.pdf",
    converter=md_ocr,
    name="ocr_sustainable_urban_mobility_trends_2025.md",
    chars=1200,
)

**`data\pdfs\sustainable_urban_mobility_trends_2025.pdf`** -> `out\ocr_sustainable_urban_mobility_trends_2025.md` (4,127 chars)

---

## Page 1

*[Image OCR]
No visible text.
[End OCR]*

Sustainable Urban

Mobility Trends 2025

This presentation offers urban planners insights into sustainable urban

mobility. We will explore key trends and strategies. Our focus areas include

EV adoption, micromobility, and smart infrastructure. Our goal is to guide

future urban development.

*[Image OCR]
M
[End OCR]*

by Muhammad Cikal Merdeka

*[Image OCR]
Made with **GAMMA**
[End OCR]*

## Page 2

Key Challenges in Urban Mobility

Traffic Air PollutionLimited Aging

CongestionAccessibilityInfrastructure

Transportation causes 27%

US cities lose $166 billion Marginalized communities A $4.5 trillion investment is of US GHG emissions. This

annually. Traffic impacts often lack transit options. needed by 2025. harms public health.

productivity and quality of This creates social inequity.Modernization is critical for

life.efficiency.

*[Image OCR]
Made with **GAMMA**
[End OCR]*

## Page 3

*[Image OCR]
MaaS
[End OCR]*

Emerging Technologies

Electric Vehicles Micromobility

(EVs)

E-scooters and e-bikes cut

Global market projected at

congestion by 10%. They

$800 billion by 2027. EVs offer flexible urban travel.

reduce tailp

> ... 2,927 more characters (full text saved in `out/`)

"## Page 1\n\n*[Image OCR]\nNo visible text.\n[End OCR]*\n\nSustainable Urban\n\nMobility Trends 2025\n\nThis presentation offers urban planners insights into sustainable urban\n\nmobility. We will explore key trends and strategies. Our focus areas include\n\nEV adoption, micromobility, and smart infrastructure. Our goal is to guide\n\nfuture urban development.\n\n*[Image OCR]\nM\n[End OCR]*\n\nby Muhammad Cikal Merdeka\n\n*[Image OCR]\nMade with **GAMMA**\n[End OCR]*\n\n## Page 2\n\nKey Challenges in Urban Mobility\n\nTraffic Air PollutionLimited Aging\n\nCongestionAccessibilityInfrastructure\n\nTransportation causes 27%\n\nUS cities lose $166 billion Marginalized communities A $4.5 trillion investment is of US GHG emissions. This\n\nannually. Traffic impacts often lack transit options. needed by 2025. harms public health.\n\nproductivity and quality of This creates social inequity.Modernization is critical for\n\nlife.efficiency.\n\n*[Image OCR]\nMade with **GAMMA**\n[End OCR]*\n\n##

In [6]:
# Using the perkembangan_transportasi_agustus_2026.pdf file.
convert_and_show(
    DATA / "pdfs" / "perkembangan_transportasi_agustus_2026.pdf",
    converter=md_ocr,
    name="ocr_2_perkembangan_transportasi_agustus_2026.md",
    chars=1200,
)

**`data\pdfs\perkembangan_transportasi_agustus_2026.pdf`** -> `out\ocr_2_perkembangan_transportasi_agustus_2026.md` (11,640 chars)

---

## Page 1

*[Image OCR]
![Logo]
[End OCR]*

BADAN PUSAT STATISTIK

KOTA PALU

*[Image OCR]
![QR code](data:image/png;base64, image)
[End OCR]*

B. Perkembangan Transportasi Kota Palu

	Jumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu

selama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan

Juli 2026.

	Volume barang yang dibongkar melalui bandar udara tercatat 979.186 ton atau

turun sebesar 8,65 persen, sementara itu barang yang dimuat tercatat 495.693

ton mengalami penurunan sebesar 37,43 persen dibandingkan Juli 2026.

	Pada Agustus 2026, Jumlah penumpang melalui Pelabuhan Pantoloan tercatat

8.016 orang.

	Volume barang yang dibongkar melalui Pelabuhan Pantoloan sebesar 10.653

ton, sementara volume barang yang dimuat tidak ada.

	Jumlah petikemas yang dibongkar melalui pelabuhan pantoloan sebesar 4.527

box sementara jumlah petikemas yang dimuat sebesar 1.570

	Frekuensi pelayaran yang beroperasi di seluruh wilayah Kota Palu, tercatat 486

kali.

## Page 2

Perkembangan Tingkat Penggunaan Sarana Transportasi Kota Palu Agustus 2026
BRS No. 31/10/7271/Th. VIII, 1 Agustus 2026
1. Angkutan Udara
Pada Agustus 2026, terdap

> ... 10,440 more characters (full text saved in `out/`)

'## Page 1\n\n*[Image OCR]\n![Logo]\n[End OCR]*\n\nBADAN PUSAT STATISTIK\n\nKOTA PALU\n\n*[Image OCR]\n![QR code](data:image/png;base64, image)\n[End OCR]*\n\nB. Perkembangan Transportasi Kota Palu\n\n\x84\tJumlah aktivitas penumpang melalui Bandar Udara Mutiara Sis Aljufri di Kota Palu\n\nselama Agustus 2026 tercatat 61.958 orang atau naik 2,11 persen dibandingkan\n\nJuli 2026.\n\n\x84\tVolume barang yang dibongkar melalui bandar udara tercatat 979.186 ton atau\n\nturun sebesar 8,65 persen, sementara itu barang yang dimuat tercatat 495.693\n\nton mengalami penurunan sebesar 37,43 persen dibandingkan Juli 2026.\n\n\x84\tPada Agustus 2026, Jumlah penumpang melalui Pelabuhan Pantoloan tercatat\n\n8.016 orang.\n\n\x84\tVolume barang yang dibongkar melalui Pelabuhan Pantoloan sebesar 10.653\n\nton, sementara volume barang yang dimuat tidak ada.\n\n\x84\tJumlah petikemas yang dibongkar melalui pelabuhan pantoloan sebesar 4.527\n\nbox sementara jumlah petikemas yang dimuat sebesar 1.570\n\n\

## PPT

Slide decks convert to Markdown with one section per slide. With an `llm_client` configured, embedded pictures additionally get an LLM-generated description — that mode is optional below since it costs one call per picture.

In [10]:
pptx = DATA / "powerpoints" / "sustainable_urban_mobility_trends_2025.pptx"
convert_and_show(pptx, name=pptx.stem + ".md", chars=1500)

**`data\powerpoints\sustainable_urban_mobility_trends_2025.pptx`** -> `out\sustainable_urban_mobility_trends_2025.md` (4,228 chars)

---

<!-- Slide number: 1 -->

![preencoded.png](Image0.jpg)
Sustainable Urban Mobility Trends 2025
This presentation offers urban planners insights into sustainable urban mobility. We will explore key trends and strategies. Our focus areas include EV adoption, micromobility, and smart infrastructure. Our goal is to guide future urban development.
by Muhammad Cikal Merdeka

![preencoded.png](Image1.jpg)

<!-- Slide number: 2 -->
Key Challenges in Urban Mobility
Traffic Congestion
Air Pollution
Limited Accessibility
Aging Infrastructure
Transportation causes 27% of US GHG emissions. This harms public health.
US cities lose $166 billion annually. Traffic impacts productivity and quality of life.
Marginalized communities often lack transit options. This creates social inequity.
A $4.5 trillion investment is needed by 2025. Modernization is critical for efficiency.

<!-- Slide number: 3 -->

![preencoded.png](Image0.jpg)
Emerging Technologies

![preencoded.png](Image1.jpg)

![preencoded.png](Image2.jpg)
Electric Vehicles (EVs)
Micromobility
E-scooters and e-bikes cut congestion by 10%. They offer flexible urban travel.
Global market projected at $800 billion by 2027. EVs reduce tailpipe emissions.

![preencoded.png](Image3.jpg)

![preencoded.png](Image4.jpg)
Smart Infrastructure
MaaS (Mobility-as-a-Service)
Sensors and data optimize traffic flow. This improves efficiency and safety.
Integrated platforms reduce car usage by 20%. They streamline transportation access.

<!-- Slide number

> ... 2,728 more characters (full text saved in `out/`)

"<!-- Slide number: 1 -->\n\n![preencoded.png](Image0.jpg)\nSustainable Urban Mobility Trends 2025\nThis presentation offers urban planners insights into sustainable urban mobility. We will explore key trends and strategies. Our focus areas include EV adoption, micromobility, and smart infrastructure. Our goal is to guide future urban development.\nby Muhammad Cikal Merdeka\n\n![preencoded.png](Image1.jpg)\n\n<!-- Slide number: 2 -->\nKey Challenges in Urban Mobility\nTraffic Congestion\nAir Pollution\nLimited Accessibility\nAging Infrastructure\nTransportation causes 27% of US GHG emissions. This harms public health.\nUS cities lose $166 billion annually. Traffic impacts productivity and quality of life.\nMarginalized communities often lack transit options. This creates social inequity.\nA $4.5 trillion investment is needed by 2025. Modernization is critical for efficiency.\n\n<!-- Slide number: 3 -->\n\n![preencoded.png](Image0.jpg)\nEmerging Technologies\n\n![preencoded.png](Image1.

In [4]:
pptx = DATA / "powerpoints" / "june_2024_sales_report.pptx"
convert_and_show(pptx, name=pptx.stem + ".md", chars=1500)

**`data\powerpoints\june_2024_sales_report.pptx`** -> `out\june_2024_sales_report.md` (863 chars)

---

<!-- Slide number: 1 -->

ACME ECOMMERCE  •  MONTHLY REPORT

June 2024 Sales Report
Ecommerce performance review — revenue, orders, and top products
Prepared by the Sales Team  •  Internal use

<!-- Slide number: 2 -->
Key Metrics — June 2024

$63,000
Total Revenue
1,620
Orders
$38.89
Avg. Order Value
+10.5%
Revenue vs. May

![slide_daily.png](Picture7.jpg)

<!-- Slide number: 3 -->
Sales Performance

![slide_weekly.png](Picture3.jpg)

![slide_categories.png](Picture4.jpg)

<!-- Slide number: 4 -->
Top Products & Takeaways

![slide_top_products.png](Picture3.jpg)
•  Revenue up 10.5% MoM — $63.0K in June vs. $57.0K in May.
•  Electronics dominated — 35% of revenue, led by Earbuds ($6.9K).
•  Strong finish — Week 4 hit $17.6K on the end-of-month promo.
•  Growth opportunity — Sports & Beauty are the smallest segments; good candidates for July campaigns.

'<!-- Slide number: 1 -->\n\nACME ECOMMERCE  •  MONTHLY REPORT\n\nJune 2024 Sales Report\nEcommerce performance review — revenue, orders, and top products\nPrepared by the Sales Team  •  Internal use\n\n<!-- Slide number: 2 -->\nKey Metrics — June 2024\n\n$63,000\nTotal Revenue\n1,620\nOrders\n$38.89\nAvg. Order Value\n+10.5%\nRevenue vs. May\n\n![slide_daily.png](Picture7.jpg)\n\n<!-- Slide number: 3 -->\nSales Performance\n\n![slide_weekly.png](Picture3.jpg)\n\n![slide_categories.png](Picture4.jpg)\n\n<!-- Slide number: 4 -->\nTop Products & Takeaways\n\n![slide_top_products.png](Picture3.jpg)\n•  Revenue up 10.5% MoM — $63.0K in June vs. $57.0K in May.\n•  Electronics dominated — 35% of revenue, led by Earbuds ($6.9K).\n•  Strong finish — Week 4 hit $17.6K on the end-of-month promo.\n•  Growth opportunity — Sports & Beauty are the smallest segments; good candidates for July campaigns.'

In [7]:
# Optional: same deck, but also describe slide images with the LLM
pptx = DATA / "powerpoints" / "sustainable_urban_mobility_trends_2025.pptx"
RUN_PPTX_LLM = True  # set True to spend extra API calls on slide imagery
if RUN_PPTX_LLM:
    convert_and_show(pptx, converter=md_llm, prompt=OCR_PROMPT, name=f"llm_{pptx.stem}.md", chars=1200)

**`data\powerpoints\sustainable_urban_mobility_trends_2025.pptx`** -> `out\llm_sustainable_urban_mobility_trends_2025.md` (4,736 chars)

---

<!-- Slide number: 1 -->

![L&FICE preencoded.png](Image0.jpg)
Sustainable Urban Mobility Trends 2025
This presentation offers urban planners insights into sustainable urban mobility. We will explore key trends and strategies. Our focus areas include EV adoption, micromobility, and smart infrastructure. Our goal is to guide future urban development.
by Muhammad Cikal Merdeka

![# M preencoded.png](Image1.jpg)

<!-- Slide number: 2 -->
Key Challenges in Urban Mobility
Traffic Congestion
Air Pollution
Limited Accessibility
Aging Infrastructure
Transportation causes 27% of US GHG emissions. This harms public health.
US cities lose $166 billion annually. Traffic impacts productivity and quality of life.
Marginalized communities often lack transit options. This creates social inequity.
A $4.5 trillion investment is needed by 2025. Modernization is critical for efficiency.

<!-- Slide number: 3 -->

![Maas Google this Stalings Integrated Sensors preencoded.png](Image0.jpg)
Emerging Technologies

![preencoded.png](Image1.jpg)

![No text visible. preencoded.png](Image2.jpg)
Electric Vehicles (EVs)
Micromobility
E-scooters and e-bikes cut congestion by 10%. They offer flexible urban travel.

> ... 3,536 more characters (full text saved in `out/`)

In [5]:
# Optional: same deck, but also describe slide images with the LLM
pptx = DATA / "powerpoints" / "june_2024_sales_report.pptx"
RUN_PPTX_LLM = True  # set True to spend extra API calls on slide imagery
if RUN_PPTX_LLM:
    convert_and_show(pptx, converter=md_llm, prompt=OCR_PROMPT, name=f"llm_{pptx.stem}.md", chars=1200)

**`data\powerpoints\june_2024_sales_report.pptx`** -> `out\llm_june_2024_sales_report.md` (2,231 chars)

---

<!-- Slide number: 1 -->

ACME ECOMMERCE  •  MONTHLY REPORT

June 2024 Sales Report
Ecommerce performance review — revenue, orders, and top products
Prepared by the Sales Team  •  Internal use

<!-- Slide number: 2 -->
Key Metrics — June 2024

$63,000
Total Revenue
1,620
Orders
$38.89
Avg. Order Value
+10.5%
Revenue vs. May

![# Daily Revenue — June 2024 Peak day: Jun 21 — $2,799 | Date | Revenue | |---|---:| | Jun 1 | $2,018 | | Jun 2 | $1,811 | | Jun 3 | $2,410 | | Jun 4 | $1,718 | | Jun 5 | $2,272 | | Jun 6 | $2,068 | | Jun 7 | $1,700 | | Jun 8 | $2,238 | | Jun 9 | $1,676 | | Jun 10 | $2,150 | | Jun 11 | $1,715 | | Jun 12 | $1,739 | | Jun 13 | $2,139 | | Jun 14 | $2,620 | | Jun 15 | $1,779 | | Jun 16 | $1,898 | | Jun 17 | $2,381 | | Jun 18 | $2,764 | | Jun 19 | $2,321 | | Jun 20 | $2,105 | | Jun 21 | $2,799 | | Jun 22 | $1,687 | | Jun 23 | $2,658 | | Jun 24 | $1,977 | | Jun 25 | $1,804 | | Jun 26 | $1,772 | | Jun 27 | $2,000 | | Jun 28 | $2,607 | | Jun 29 | $1,847 | | Jun 30 | $2,327 | - Y-axis labels: $0K, $0K, $1K, $2K, $2K, $3K, $4K - X-axis labels: Jun 2, Jun 4, Jun 6, Jun 8, Jun 10, Jun 12, Jun 14, Jun 16, Jun 18, Jun 20, Jun 22, Jun 24, Jun 26, Jun 28, Jun 30 slide_daily.p

> ... 1,031 more characters (full text saved in `out/`)

### markitdown-ocr plugin (PPT)

The plugin's PPTX converter handles picture shapes, image placeholders, and grouped images, asking the LLM for a description first and using OCR as the fallback when no description is returned. Core shape ordering, tables, charts, and speaker notes are preserved.

With **30 unique images** this deck is the costliest target — deliberately behind a flag.

In [8]:
RUN_PPTX_PLUGIN_OCR = True  # ~30 vision calls on this deck
if RUN_PPTX_PLUGIN_OCR:
    convert_and_show(
        pptx,
        converter=md_ocr,
        name=f"ocr_pptx_{pptx.stem}.md",
        chars=1400,
    )

**`data\powerpoints\sustainable_urban_mobility_trends_2025.pptx`** -> `out\ocr_pptx_sustainable_urban_mobility_trends_2025.md` (5,108 chars)

---

<!-- Slide number: 1 -->

![L&A TOWER L&FICe preencoded.png](Image0.jpg)
Sustainable Urban Mobility Trends 2025
This presentation offers urban planners insights into sustainable urban mobility. We will explore key trends and strategies. Our focus areas include EV adoption, micromobility, and smart infrastructure. Our goal is to guide future urban development.
by Muhammad Cikal Merdeka

![M preencoded.png](Image1.jpg)

<!-- Slide number: 2 -->
Key Challenges in Urban Mobility
Traffic Congestion
Air Pollution
Limited Accessibility
Aging Infrastructure
Transportation causes 27% of US GHG emissions. This harms public health.
US cities lose $166 billion annually. Traffic impacts productivity and quality of life.
Marginalized communities often lack transit options. This creates social inequity.
A $4.5 trillion investment is needed by 2025. Modernization is critical for efficiency.

<!-- Slide number: 3 -->

![MaaS Google this Settings Chat 1:41 Map 1:49 Full 1:43 Integrated Sensors 1 2 3 2 6 preencoded.png](Image0.jpg)
Emerging Technologies

![<!-- No text visible --> preencoded.png](Image1.jpg)

*[Image OCR]
The image contains no readable text.
[End OCR]*
Electric Vehicles (EVs)
Micromobility
E-scooters and e-bikes cut congestion by 10%. They offer flexible urban travel.
Global market projected at $800 billion by 2027. EVs reduce tailpipe emissions.

*[Image OCR]
The image contains 

> ... 3,708 more characters (full text saved in `out/`)

### PPTX pages → LLM (via PDF export)

Same physics as DOCX: PPTX stores no committed pagination, so the render-first trick needs a layout engine — **MS PowerPoint via COM**, with LibreOffice headless as the fallback. `pptx_to_pdf()` below auto-detects both and errors with instructions otherwise.

Why bother when `md_llm` can touch pptx images? The built-in pptx route sends each **embedded picture** while the converter handles native text separately — fine when slides are text-rich, but when slides are basically rendered posters (like `june_2024_sales_report.pptx`: 4 slides, 4 images, ~860 chars of native text) there is no layout context and descriptions get vague. Exporting to PDF and OCRing whole slides preserves the composed visual exactly — the same advantage you saw on the BPS report's PDF twin.

Flow: `SALES_DECK` → export to `out/pptx_pdf/` → render slides → OCR each (one call per slide ≈ 4) → stitch to `out/scanned_pptx_<stem>.md`. Needs `vision_to_markdown()` (Scanned-PDF section) — run that first.

In [9]:
# PPTX -> PDF export with auto-detected layout engine (PowerPoint first, LibreOffice fallback)
def pptx_to_pdf(pptx_path, out_dir=None):
    pptx_path = Path(pptx_path)
    out_dir = Path(out_dir or OUT / "pptx_pdf")
    out_dir.mkdir(exist_ok=True, parents=True)
    dest = out_dir / (pptx_path.stem + ".pdf")

    com_error = None
    try:
        import win32com.client  # pywin32, ships with the docx2pdf dependency

        app = win32com.client.Dispatch("PowerPoint.Application")
        try:
            pres = app.Presentations.Open(str(pptx_path.resolve()), True, False, False)
            try:
                pres.SaveAs(str(dest.resolve()), 32)  # 32 = ppSaveAsPDF
            finally:
                pres.Close()
        finally:
            app.Quit()
        if dest.exists():
            return dest
    except Exception as e:
        com_error = e

    soffice = shutil.which("soffice") or shutil.which("soffice.exe")
    if soffice:
        subprocess.run(
            [soffice, "--headless", "--norestore", "--convert-to", "pdf", "--outdir", str(out_dir), str(pptx_path)],
            check=True,
            capture_output=True,
            timeout=600,
        )
        if dest.exists():
            return dest

    com_note = f" (COM error: {type(com_error).__name__}: {com_error})" if com_error else ""
    raise RuntimeError(
        f"No layout engine available to render '{pptx_path.name}' to PDF.{com_note}\n"
        "- Option A: install MS PowerPoint (COM path succeeds automatically)\n"
        "- Option B: install LibreOffice (put soffice.exe on PATH)\n"
        "- Option C: export the PDF yourself (PowerPoint / LibreOffice / Google Slides)\n"
        f"  and save it as: {dest} - the next cell picks it up."
    )


SALES_DECK = DATA / "powerpoints" / "june_2024_sales_report.pptx"
sales_pdf_path = pptx_to_pdf(SALES_DECK)
print("engine used: auto-detected -> exported", sales_pdf_path.name, f"({sales_pdf_path.stat().st_size / 1024:.0f} KB)")

engine used: auto-detected -> exported june_2024_sales_report.pdf (471 KB)


In [11]:
def vision_to_markdown(image_path, prompt=OCR_PROMPT, *, model=MODEL, effort=REASONING_EFFORT):
    b64 = base64.b64encode(Path(image_path).read_bytes()).decode("utf-8")
    ext = Path(image_path).suffix.lstrip(".").lower()
    mime = "jpeg" if ext == "jpg" else ext
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}},
                ],
            }
        ],
        extra_body={"reasoning_effort": effort},
    )
    return response.choices[0].message.content or ""

In [12]:
# Render the exported deck's slides, OCR each with the direct vision call
# (reasoning_effort="none"), and stitch -> out/scanned_pptx_<stem>.md
PPTX_PAGES_DIR = OUT / "pptx_pages"
PPTX_PAGES_DIR.mkdir(exist_ok=True, parents=True)

doc = pymupdf.open(sales_pdf_path)
sales_pages = []
for i, page in enumerate(doc, start=1):
    png = PPTX_PAGES_DIR / f"{sales_pdf_path.stem}_p{i:02d}.png"
    page.get_pixmap(dpi=150).save(png)
    sales_pages.append(png)
doc.close()

sales_pages_md = []
for png in sales_pages:
    print("ocr:", png.name, "...", end=" ")
    t0 = time.perf_counter()
    text = vision_to_markdown(png).strip()
    print(f"{len(text):,} chars in {time.perf_counter() - t0:.1f}s")
    sales_pages_md.append(f"<!-- slide {png.stem[-2:]} -->\n\n{text}")

sales_stitched = "\n\n---\n\n".join(sales_pages_md)
sales_stitched_path = save_markdown(f"scanned_pptx_{sales_pdf_path.stem}.md", sales_stitched)
SUMMARY.append((str(SALES_DECK), "llm-scanned (pptx via pdf)", len(sales_stitched)))
display(Markdown(f"**`{SALES_DECK.name}`** (via PDF export) -> `{sales_stitched_path.name}` ({len(sales_stitched):,} chars)\n\n---"))
display(Markdown(preview(sales_stitched, 900)))

ocr: june_2024_sales_report_p01.png ... 166 chars in 3.0s
ocr: june_2024_sales_report_p02.png ... 802 chars in 4.2s
ocr: june_2024_sales_report_p03.png ... 318 chars in 2.7s
ocr: june_2024_sales_report_p04.png ... 549 chars in 2.6s


**`june_2024_sales_report.pptx`** (via PDF export) -> `scanned_pptx_june_2024_sales_report.md` (1,932 chars)

---

<!-- slide 01 -->

ACME ECOMMERCE • MONTHLY REPORT

# June 2024 Sales Report

Ecommerce performance review — revenue, orders, and top products

Prepared by the Sales Team • Internal use

---

<!-- slide 02 -->

# Key Metrics — June 2024

$63,000  
Total Revenue

1,620  
Orders

$38.89  
Avg. Order Value

+10.5%  
Revenue vs. May

## Daily Revenue — June 2024

**Peak day: Jun 21 — $2,799**

| Date | Revenue |
|---|---:|
| Jun 1 | $2,018 |
| Jun 2 | $1,811 |
| Jun 3 | $2,410 |
| Jun 4 | $1,718 |
| Jun 5 | $2,272 |
| Jun 6 | $2,068 |
| Jun 7 | $1,700 |
| Jun 8 | $2,238 |
| Jun 9 | $1,676 |
| Jun 10 | $2,150 |
| Jun 11 | $1,715 |
| Jun 12 | $1,739 |
| Jun 13 | $2,139 |
| Jun 14 | $2,620 |
| Jun 15 | $1,779 |
| Jun 16 | $1,898 |
| Jun 17 | $2,381 |
| Jun 18 | $2,764 |
| Jun 19 | $2,321 |
| Jun 20 | $2,105 |
| Jun 21 | $2,799 |
| Jun 22 | $1,687 |
| Jun 23 | $2,658 |
| Jun 24 | $1,977 |
| Jun 

> ... 1,032 more characters (full text saved in `out/`)

## Excel

Workbooks come out as Markdown, with one pipe-separated table per worksheet — handy for feeding tabular data to LLMs.

In [12]:
xlsx = DATA / "spreadsheets" / "example_table.xlsx"
convert_and_show(xlsx, name=xlsx.stem + ".md", chars=1600)

**`data\spreadsheets\example_table.xlsx`** -> `out\example_table.md` (21,071 chars)

---

## Sheet1
| Unnamed: 0 | Code | Amount Raw | Amount Processed | Order ID | Month |
| --- | --- | --- | --- | --- | --- |
| 0 | 5.502600e+11 | 64000 | 62720.00 | TRD2022122305306 | January |
| 1 | 5.502600e+11 | 58092 | 56930.16 | TRD2022122500888 | January |
| 2 | 5.502600e+11 | 43062 | 42200.76 | TRD2022122500830 | January |
| 3 | 5.502600e+11 | 489000 | 479220.00 | TRD2022122606284 | January |
| 4 | 5.502600e+11 | 10000 | 9800.00 | TRD2022122701823 | January |
| 5 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706644 | January |
| 6 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706674 | January |
| 7 | 5.502600e+11 | 48080 | 47118.40 | TRD2022122801939 | January |
| 8 | 5.502600e+11 | 149274 | 146288.52 | TRD2022122802089 | January |
| 9 | 5.502600e+11 | 110000 | 107800.00 | TRD2022122802390 | January |
| 10 | 5.502600e+11 | 34000 | 33320.00 | TRD2022122907187 | January |
| 11 | 5.502600e+11 | 116200 | 113876.00 | TRD2022122801897 | January |
| 12 | 5.502600e+11 | 271510 | 266079.80 | TRD2022123004626 | January |
| 13 | 5.502600e+11 | 110000 | 107800.00 | TRD2023010204762 | January |
| 14 | 5.502600e+11 | 44000 | 43120.00 | TRD2023010206759 | January |
| 15 | 5.502600e+11 | 111000 | 108780.00 | TRD2023010301543 | January |
| 16 | 5.502600e+11 | 50000 | 49000.00 | TRD2023010301562 | January |
| 17 | 5.502600e+11 | 59000 | 57820.00 | TRD2023010308689 | January |
| 18 | 5.502600e+11 | 36000 | 35280.00 | TRD2023010409180 | January |
| 19 | 5.502600e+11 | 64000 | 62720.00 | TRD2023010510977 | January |
| 20 | 5.502600e+11 | 140000 | 137200.00 | TRD2023010701418 | January |
| 21

> ... 19,471 more characters (full text saved in `out/`)

'## Sheet1\n| Unnamed: 0 | Code | Amount Raw | Amount Processed | Order ID | Month |\n| --- | --- | --- | --- | --- | --- |\n| 0 | 5.502600e+11 | 64000 | 62720.00 | TRD2022122305306 | January |\n| 1 | 5.502600e+11 | 58092 | 56930.16 | TRD2022122500888 | January |\n| 2 | 5.502600e+11 | 43062 | 42200.76 | TRD2022122500830 | January |\n| 3 | 5.502600e+11 | 489000 | 479220.00 | TRD2022122606284 | January |\n| 4 | 5.502600e+11 | 10000 | 9800.00 | TRD2022122701823 | January |\n| 5 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706644 | January |\n| 6 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706674 | January |\n| 7 | 5.502600e+11 | 48080 | 47118.40 | TRD2022122801939 | January |\n| 8 | 5.502600e+11 | 149274 | 146288.52 | TRD2022122802089 | January |\n| 9 | 5.502600e+11 | 110000 | 107800.00 | TRD2022122802390 | January |\n| 10 | 5.502600e+11 | 34000 | 33320.00 | TRD2022122907187 | January |\n| 11 | 5.502600e+11 | 116200 | 113876.00 | TRD2022122801897 | January |\n| 12 | 5.502600e+11 | 2715

In [13]:
# Same output in raw form: see the real pipe-table syntax markitdown produced
print((OUT / (xlsx.stem + ".md")).read_text(encoding="utf-8")[:2000])

## Sheet1
| Unnamed: 0 | Code | Amount Raw | Amount Processed | Order ID | Month |
| --- | --- | --- | --- | --- | --- |
| 0 | 5.502600e+11 | 64000 | 62720.00 | TRD2022122305306 | January |
| 1 | 5.502600e+11 | 58092 | 56930.16 | TRD2022122500888 | January |
| 2 | 5.502600e+11 | 43062 | 42200.76 | TRD2022122500830 | January |
| 3 | 5.502600e+11 | 489000 | 479220.00 | TRD2022122606284 | January |
| 4 | 5.502600e+11 | 10000 | 9800.00 | TRD2022122701823 | January |
| 5 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706644 | January |
| 6 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706674 | January |
| 7 | 5.502600e+11 | 48080 | 47118.40 | TRD2022122801939 | January |
| 8 | 5.502600e+11 | 149274 | 146288.52 | TRD2022122802089 | January |
| 9 | 5.502600e+11 | 110000 | 107800.00 | TRD2022122802390 | January |
| 10 | 5.502600e+11 | 34000 | 33320.00 | TRD2022122907187 | January |
| 11 | 5.502600e+11 | 116200 | 113876.00 | TRD2022122801897 | January |
| 12 | 5.502600e+11 | 271510 | 266079.80 |

In [10]:
xlsx_2 = DATA / "spreadsheets" / "ecommerce_sales.xlsx"
convert_and_show(xlsx_2, name=xlsx_2.stem + ".md", chars=1600)

**`data\spreadsheets\ecommerce_sales.xlsx`** -> `out\ecommerce_sales.md` (610 chars)

---

## Sales Report
| Ecommerce Sales Report – H1 2024 | Unnamed: 1 | Unnamed: 2 | Unnamed: 3 |
| --- | --- | --- | --- |
| Month | Orders | Revenue | NaN |
| January | 1200 | 45000 | NaN |
| February | 1100 | 41000 | NaN |
| March | 1350 | 52000 | NaN |
| April | 1280 | 48000 | NaN |
| May | 1500 | 57000 | NaN |
| June | 1620 | 63000 | NaN |
| NaN | NaN | NaN | NaN |
| Product | Category | Units Sold | Revenue |
| Wireless Mouse | Electronics | 850 | 12750 |
| Yoga Mat | Sports | 720 | 9360 |
| Coffee Maker | Home | 640 | 19200 |
| Backpack | Fashion | 590 | 11800 |
| Earbuds | Electronics | 1540 | 23100 |

'## Sales Report\n| Ecommerce Sales Report – H1 2024 | Unnamed: 1 | Unnamed: 2 | Unnamed: 3 |\n| --- | --- | --- | --- |\n| Month | Orders | Revenue | NaN |\n| January | 1200 | 45000 | NaN |\n| February | 1100 | 41000 | NaN |\n| March | 1350 | 52000 | NaN |\n| April | 1280 | 48000 | NaN |\n| May | 1500 | 57000 | NaN |\n| June | 1620 | 63000 | NaN |\n| NaN | NaN | NaN | NaN |\n| Product | Category | Units Sold | Revenue |\n| Wireless Mouse | Electronics | 850 | 12750 |\n| Yoga Mat | Sports | 720 | 9360 |\n| Coffee Maker | Home | 640 | 19200 |\n| Backpack | Fashion | 590 | 11800 |\n| Earbuds | Electronics | 1540 | 23100 |'

### markitdown-ocr plugin (XLSX)

Workbook images are OCR'd from the repaired workbook and listed under a `### Images in this sheet:` heading after each sheet's table. `example_table.xlsx` has no images (identical-to-plain output; zero calls), and the added `ecommerce_sales.xlsx` holds **2 unique images** — its plugin cell costs ~2 calls when its flag is on. Legacy `.xls` files stay on the core converter, without image OCR.

In [9]:
RUN_XLSX_PLUGIN_OCR = True  # free today (0 images); opt in for image-bearing versions
xlsx_ocr = DATA / "spreadsheets" / "ecommerce_sales.xlsx"
if RUN_XLSX_PLUGIN_OCR:
    convert_and_show(
        xlsx_ocr,
        converter=md_ocr,
        name=f"ocr_{xlsx_ocr.stem}.md",
        chars=1600,
    )

**`data\spreadsheets\ecommerce_sales.xlsx`** -> `out\ocr_ecommerce_sales.md` (1,065 chars)

---

## Sales Report
| Ecommerce Sales Report – H1 2024 | Unnamed: 1 | Unnamed: 2 | Unnamed: 3 |
| --- | --- | --- | --- |
| Month | Orders | Revenue | NaN |
| January | 1200 | 45000 | NaN |
| February | 1100 | 41000 | NaN |
| March | 1350 | 52000 | NaN |
| April | 1280 | 48000 | NaN |
| May | 1500 | 57000 | NaN |
| June | 1620 | 63000 | NaN |
| NaN | NaN | NaN | NaN |
| Product | Category | Units Sold | Revenue |
| Wireless Mouse | Electronics | 850 | 12750 |
| Yoga Mat | Sports | 720 | 9360 |
| Coffee Maker | Home | 640 | 19200 |
| Backpack | Fashion | 590 | 11800 |
| Earbuds | Electronics | 1540 | 23100 |

### Images in this sheet:

*[Image OCR]
# Monthly Revenue (H1 2024)

| Month | Revenue ($) |
|---|---:|
| January | $45,000 |
| February | $41,000 |
| March | $52,000 |
| April | $48,000 |
| May | $57,000 |
| June | $63,000 |

Revenue ($): 0, 10000, 20000, 30000, 40000, 50000, 60000, 70000
[End OCR]*

*[Image OCR]
# Revenue Share by Product

- Earbuds: 30.3%
- Backpack: 15.5%
- Coffee Maker: 25.2%
- Yoga Mat: 12.3%
- Wireless Mouse: 16.7%
[End OCR]*

## Recap

What each section produced — source, converter used, and markdown size. Full outputs are in `out/`.

In [14]:
# Recap table
if SUMMARY:
    seen = set()
    rows = ["| source | converter | md chars |", "|---|---|---:|"]
    for source, kind, n in SUMMARY:
        key = (Path(source).name, kind)
        if key in seen:
            continue
        seen.add(key)
        rows.append(f"| `{Path(source).name}` | {kind} | {n:,} |")
    display(Markdown("\n".join(rows)))

| source | converter | md chars |
|---|---|---:|
| `sample_text.jpg` | plain | 0 |
| `sample_image_1.png` | plain | 0 |
| `sample_text.jpg` | llm | 65 |
| `sample_image_1.png` | llm | 1,409 |
| `page_2.png` | llm | 1,556 |
| `docling_technical_report.pdf` | plain | 84,201 |
| `FinancialStatement-2025-III-BBRI.pdf` | plain | 123,596 |
| `long_table_pdf.pdf` | plain | 51,532 |
| `pdf_with_tables_example.pdf` | plain | 22,444 |
| `Product Strategy & Decision Handbook — Atlas Pay.pdf` | plain | 8,633 |
| `sustainable_urban_mobility_trends_2025.pdf` | plain | 3,389 |
| `sustainable_urban_mobility_trends_2025.pdf` | llm-scanned | 3,653 |
| `sustainable_urban_mobility_trends_2025.pptx` | plain | 4,228 |
| `example_table.xlsx` | plain | 21,071 |